# Reporte técnico — Filtro de relevancia para X y YouTube
## Voz del Cliente más allá del NPS · Equipo 14

**Objetivo de esta entrega:** documentar, de forma reproducible, cómo se construyó y evaluó el clasificador que determina si un comentario de **X o YouTube** está realmente relacionado con la **app móvil de Banamex** antes de enviarlo a las etapas posteriores de modelado de temas y análisis de sentimiento.

### Resumen ejecutivo

Las reseñas de App Store y Google Play están asociadas directamente con la aplicación por su propia fuente. En X y YouTube no existe esa garantía: un comentario puede mencionar a Banamex y hablar de una sucursal, otro producto, otra aplicación, una promoción o un tema completamente distinto.

Por ello se agregó una capa de **prefiltrado de relevancia**:

**X / YouTube → embeddings MiniLM → clasificador de relevancia → reseñas seleccionadas → tópicos y sentimiento**

El experimento comparó un control basado únicamente en la plataforma contra dos clasificadores lineales entrenados sobre embeddings MiniLM de **384 dimensiones**:

| Modelo | Accuracy | Balanced Accuracy | Precision relevant | Recall relevant | F1 relevant |
|---|---:|---:|---:|---:|---:|
| Platform-only baseline | 0.815 | 0.835 | 0.938 | 0.784 | 0.854 |
| Logistic Regression | **0.905** | 0.878 | 0.917 | **0.948** | **0.932** |
| Linear SVM | 0.899 | **0.890** | **0.938** | 0.914 | 0.926 |

**Conclusión principal:** ambos modelos basados en embeddings superan al control de plataforma. Esto aporta evidencia de que la representación MiniLM contiene señal semántica útil para distinguir relevancia.

Para una primera integración se propone **Linear SVM como modelo candidato v1**, porque el filtro está antes de otras etapas y conviene reducir falsos positivos. Sin embargo, esta selección sigue siendo provisional: el ground truth de X contiene muy pocos ejemplos irrelevantes y debe reforzarse antes de considerar el modelo definitivo.


## 1. Ubicación del filtro dentro de la arquitectura

La arquitectura aprobada separa las reseñas de tiendas de aplicaciones de las fuentes externas. App Store y Google Play continúan sin un prefiltrado adicional; X y YouTube pasan primero por el clasificador de relevancia.

El filtro no sustituye al análisis de sentimiento ni al modelado de temas. Su función es **proteger las etapas posteriores**, evitando que comentarios ajenos a la app contaminen las métricas, los tópicos y el dashboard.


![Arquitectura aprobada](attachment:arquitectura_relevancia.jpeg)

**Componente documentado en este notebook:** bloque naranja — *Filtro de relevancia*.


## 2. Planteamiento técnico

La tarea se formula como una clasificación binaria a partir únicamente del texto:

- `relevant` → el comentario contiene evidencia suficiente de que habla de la app móvil de Banamex.
- `irrelevant` → el comentario habla de otro producto, canal, banco o tema y no existe evidencia suficiente de relación con la app.

Durante la construcción del ground truth se conservó además:

- `uncertain` → el texto aislado no permite decidir con suficiente confianza.

Los casos `uncertain` se mantienen para futuras revisiones, pero **no se utilizan en el primer entrenamiento binario** para evitar introducir ruido artificial en la variable objetivo.


## 3. Entradas reproducibles

El experimento utiliza un release fijo del dataset y los embeddings ya publicados en S3.

- Dataset release: `b7a61a8b-27d8-46a7-b2ce-67d97268b741`
- Perfil de embeddings: `minilm-verbatim-v1`
- Dimensión: `384`
- Plataformas analizadas: `x`, `youtube`

Se fija el release para que las etiquetas, los textos y los embeddings correspondan exactamente al mismo conjunto de datos.


In [ ]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from voc import collector as co
from voc.embeddings import get_embeddings

DATASET_VERSION = "b7a61a8b-27d8-46a7-b2ce-67d97268b741"
EMBEDDING_VERSION = "latest"
PROFILE = "minilm-verbatim-v1"
PLATFORMS = ["youtube", "x"]

WORKSPACE = Path("/workspace")
LABEL_DIR = WORKSPACE / "notebooks" / "compi-pc"
MODEL_DIR = WORKSPACE / "models"

YT_LABELS_PATH = LABEL_DIR / "auditoria_relevance_text_only.csv"
X_LABELS_PATH = LABEL_DIR / "muestra_x_relevance_text_only.csv"

for path in [YT_LABELS_PATH, X_LABELS_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró {path}. "
            "El reporte necesita los archivos de etiquetas construidos durante la auditoría."
        )

print("Configuración lista.")


## 4. Carga y validación de embeddings

Primero se carga el snapshot preparado y después el bundle de embeddings correspondiente. La validación comprueba que:

1. exista un vector por cada registro;
2. el bundle corresponda al dataset seleccionado;
3. la selección sea de cobertura completa;
4. el orden de los registros y de los vectores permanezca alineado.

Después se recupera el texto original y el contexto mediante `record_id`, y finalmente se filtran X y YouTube.


In [ ]:
snapshot = co.get_dataset(
    stage="prepared",
    source="s3",
    version=DATASET_VERSION,
)

bundle = get_embeddings(
    dataset=snapshot,
    profile=PROFILE,
    version=EMBEDDING_VERSION,
    source="s3",
    allow_subset=False,
)

assert len(bundle.reviews) == bundle.vectors.shape[0]
assert len(bundle.reviews) == len(snapshot.data)
assert bundle.info["selection"]["kind"] == "full"

resolved = {
    "dataset_release_id": bundle.info["dataset"]["release_id"],
    "embedding_release_id": bundle.info["embedding_release_id"],
    "profile": bundle.info["profile"]["name"],
    "profile_id": bundle.info["profile"]["profile_id"],
    "loaded_reviews": len(bundle.reviews),
    "dimensions": bundle.vectors.shape[1],
}

display(pd.Series(resolved, name="resolved input").to_frame())


In [ ]:
context_columns = [
    "record_id",
    "platform",
    "preferred_provider",
    "text",
    "title",
    "relevance",
    "record_date",
]

all_reviews = bundle.reviews.merge(
    snapshot.data[context_columns],
    on="record_id",
    how="left",
    sort=False,
    validate="one_to_one",
    indicator=True,
)

assert all_reviews["_merge"].eq("both").all()
all_reviews = all_reviews.drop(columns="_merge").reset_index(drop=True)

assert all_reviews["record_id"].tolist() == bundle.reviews["record_id"].tolist()
assert all_reviews["vector_row"].tolist() == list(range(len(all_reviews)))

mask = all_reviews["platform"].isin(PLATFORMS).to_numpy()
positions = np.flatnonzero(mask)

df = all_reviews.iloc[positions].copy().reset_index(drop=True)
embeddings = bundle.vectors[positions]

df["embedding"] = list(embeddings)

assert len(df) == embeddings.shape[0]
assert df["record_id"].tolist() == bundle.reviews.iloc[positions]["record_id"].tolist()

print(f"Registros totales del release: {len(all_reviews):,}")
print(f"X + YouTube seleccionados: {len(df):,}")
print("Matriz de embeddings:", embeddings.shape)
print()
display(df.groupby("platform").size().rename("registros").to_frame())


### Resultado observado

En la ejecución utilizada para este experimento:

- dataset completo: **80,079 registros**;
- X: **6,553**;
- YouTube: **1,328**;
- total seleccionado: **7,881**;
- representación por comentario: **384 dimensiones**.

El clasificador no trabaja directamente con palabras o reglas de negocio: recibe la representación semántica MiniLM asociada a cada comentario.


## 5. Construcción del ground truth

### 5.1 YouTube: auditoría *text-only*

Existían 105 comentarios de YouTube con revisión previa. Se reauditaron usando **solo el texto**, porque el modelo de relevancia también recibirá únicamente ese texto.

Resultado:

- `irrelevant`: **49**
- `relevant`: **25**
- `uncertain`: **31**

Un hallazgo importante fue que varios comentarios originalmente considerados relevantes dependían del contexto del video o de la conversación. Si esa información no forma parte de la entrada del modelo, tampoco debe utilizarse para definir el ground truth del experimento.


### 5.2 X: muestra aleatoria etiquetada

Los comentarios de X estaban mayormente `unreviewed`, por lo que se seleccionó una muestra aleatoria reproducible de **100 registros** (`random_state=42`) y se aplicó el mismo criterio *text-only*.

Resultado:

- `relevant`: **91**
- `irrelevant`: **6**
- `uncertain`: **3**

Esta distribución reveló una limitación importante: la muestra contiene muy pocos negativos de X. Por ello, cualquier conclusión sobre la capacidad de rechazar comentarios irrelevantes de esa plataforma debe tratarse como preliminar.


In [ ]:
yt_labels = pd.read_csv(
    YT_LABELS_PATH,
    dtype={
        "record_id": "string",
        "etiqueta_texto": "string",
        "notas_revision": "string",
    },
)

x_labels = pd.read_csv(
    X_LABELS_PATH,
    dtype={
        "record_id": "string",
        "etiqueta_texto": "string",
        "notas_revision": "string",
    },
)

print("YouTube:")
print(yt_labels["etiqueta_texto"].value_counts(dropna=False))
print()
print("X:")
print(x_labels["etiqueta_texto"].value_counts(dropna=False))


### 5.3 Unión de etiquetas con embeddings

Las etiquetas se asocian con los embeddings mediante `record_id`. Solo se conservan `relevant` e `irrelevant` para el entrenamiento binario.

La variable objetivo queda codificada como:

- `0` = irrelevant
- `1` = relevant


In [ ]:
feature_cols = [
    "record_id",
    "platform",
    "text",
    "embedding",
    "relevance",
]

def attach_labels(label_df):
    clean_labels = label_df[
        ["record_id", "etiqueta_texto", "notas_revision"]
    ].copy()

    merged = clean_labels.merge(
        df[feature_cols],
        on="record_id",
        how="left",
        validate="one_to_one",
    )

    assert merged["embedding"].notna().all()
    return merged

yt_labeled = attach_labels(yt_labels)
x_labeled = attach_labels(x_labels)

clear_labels = ["relevant", "irrelevant"]

yt_binary = yt_labeled[
    yt_labeled["etiqueta_texto"].isin(clear_labels)
].copy()

x_binary = x_labeled[
    x_labeled["etiqueta_texto"].isin(clear_labels)
].copy()

dataset_modelo = pd.concat(
    [yt_binary, x_binary],
    ignore_index=True,
)

dataset_modelo["target"] = (
    dataset_modelo["etiqueta_texto"] == "relevant"
).astype(int)

print("Registros claros:", len(dataset_modelo))
print()
print(dataset_modelo["etiqueta_texto"].value_counts())
print()
print(dataset_modelo["platform"].value_counts())


## 6. Control de calidad antes del entrenamiento

### 6.1 Eliminación de duplicados exactos

Se detectaron textos idénticos repetidos. Aunque no existían conflictos de etiqueta, mantener copias exactas aumenta el riesgo de *data leakage*: el mismo contenido podría aparecer en entrenamiento y validación.

Por ello se conserva una sola copia de cada texto antes de medir el desempeño.


In [ ]:
duplicados = dataset_modelo[
    dataset_modelo["text"].duplicated(keep=False)
].sort_values("text")

print("Registros involucrados en duplicados:", len(duplicados))
print("Textos duplicados distintos:", duplicados["text"].nunique())

dataset_modelo_limpio = (
    dataset_modelo
    .drop_duplicates(subset=["text"], keep="first")
    .reset_index(drop=True)
    .copy()
)

print()
print("Registros antes:", len(dataset_modelo))
print("Registros después:", len(dataset_modelo_limpio))
print()
print(dataset_modelo_limpio["etiqueta_texto"].value_counts())
print()
print(dataset_modelo_limpio["platform"].value_counts())


El dataset final de modelado queda en **168 comentarios únicos**:

- 116 `relevant`
- 52 `irrelevant`
- 97 de X
- 71 de YouTube

### 6.2 Sesgo entre plataforma y etiqueta

La distribución de clases no es homogénea:

- X contiene principalmente comentarios relevantes.
- YouTube contiene una proporción mucho mayor de irrelevantes.

Esto puede inflar artificialmente las métricas si el modelo aprende señales asociadas a la fuente en lugar del concepto de relevancia.

Para cuantificar ese riesgo se construyó un baseline que **no lee el texto**:

- X → `relevant`
- YouTube → `irrelevant`

Si un modelo semántico no supera claramente este control, no tendríamos evidencia suficiente de que los embeddings estén aportando valor.


In [ ]:
tabla_plataforma_etiqueta = pd.crosstab(
    dataset_modelo_limpio["platform"],
    dataset_modelo_limpio["etiqueta_texto"],
    margins=True,
)

display(tabla_plataforma_etiqueta)

y_true = dataset_modelo_limpio["target"].to_numpy()
y_pred_platform = (
    dataset_modelo_limpio["platform"] == "x"
).astype(int).to_numpy()

baseline_metrics = {
    "Accuracy": accuracy_score(y_true, y_pred_platform),
    "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred_platform),
    "Precision relevant": precision_score(y_true, y_pred_platform),
    "Recall relevant": recall_score(y_true, y_pred_platform),
    "F1 relevant": f1_score(y_true, y_pred_platform),
}

display(
    pd.Series(baseline_metrics, name="Platform-only baseline")
    .round(3)
    .to_frame()
)


## 7. Estrategia de evaluación

Con 168 registros, un único `train_test_split` sería demasiado sensible a qué ejemplos caen al azar en prueba.

Se utiliza **5-fold cross-validation** y se estratifica por una combinación de:

`plataforma + clase`

De esta forma los cinco folds preservan, en la medida permitida por el tamaño de la muestra:

- X relevant
- X irrelevant
- YouTube relevant
- YouTube irrelevant

`cross_val_predict` genera una predicción *out-of-fold* para cada comentario: cada observación es evaluada por un modelo que no la vio durante su entrenamiento.

Se reportan:

- Accuracy
- Balanced Accuracy
- Precision de `relevant`
- Recall de `relevant`
- F1 de `relevant`
- Matriz de confusión

`Balanced Accuracy` es especialmente importante debido al desbalance observado.


In [ ]:
X_modelo = np.vstack(
    dataset_modelo_limpio["embedding"].to_numpy()
)

y_modelo = (
    dataset_modelo_limpio["target"]
    .astype(int)
    .to_numpy()
)

estratos = (
    dataset_modelo_limpio["platform"].astype(str)
    + "_"
    + dataset_modelo_limpio["target"].astype(str)
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_splits = list(
    cv.split(X_modelo, estratos)
)

print("Shape X:", X_modelo.shape)
print("Shape y:", y_modelo.shape)
print()
print("Estratos:")
print(estratos.value_counts())
print()
print("Folds:", len(cv_splits))


## 8. Modelo 1 — Logistic Regression

La primera alternativa es una regresión logística lineal sobre las 384 dimensiones del embedding.

El pipeline incluye:

1. `StandardScaler`
2. `LogisticRegression`
3. `class_weight="balanced"`

El balanceo de clases evita que la clase mayoritaria domine la función de decisión.


In [ ]:
modelo_logreg = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ]
)

pred_logreg = cross_val_predict(
    modelo_logreg,
    X_modelo,
    y_modelo,
    cv=cv_splits,
    method="predict",
)

prob_logreg = cross_val_predict(
    modelo_logreg,
    X_modelo,
    y_modelo,
    cv=cv_splits,
    method="predict_proba",
)[:, 1]

metrics_logreg = {
    "Accuracy": accuracy_score(y_modelo, pred_logreg),
    "Balanced Accuracy": balanced_accuracy_score(y_modelo, pred_logreg),
    "Precision relevant": precision_score(y_modelo, pred_logreg),
    "Recall relevant": recall_score(y_modelo, pred_logreg),
    "F1 relevant": f1_score(y_modelo, pred_logreg),
}

display(pd.Series(metrics_logreg, name="Logistic Regression").round(3).to_frame())

print("Confusion matrix:")
print(confusion_matrix(y_modelo, pred_logreg))


### Resultado esperado de Logistic Regression

- Accuracy: **0.905**
- Balanced Accuracy: **0.878**
- Precision relevant: **0.917**
- Recall relevant: **0.948**
- F1 relevant: **0.932**
- Matriz de confusión: `[[42, 10], [6, 110]]`

Su principal fortaleza es el **recall**: recupera casi todos los comentarios relevantes. El costo es dejar pasar más falsos positivos que Linear SVM.


## 9. Modelo 2 — Linear SVM

La segunda alternativa utiliza una frontera lineal de máxima separación (`LinearSVC`) sobre la misma representación MiniLM.

Se mantiene el mismo escalado, balanceo de clases y exactamente los mismos folds para que la comparación sea justa.


In [ ]:
modelo_svm = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LinearSVC(
                class_weight="balanced",
                random_state=42,
                max_iter=10000,
            ),
        ),
    ]
)

pred_svm = cross_val_predict(
    modelo_svm,
    X_modelo,
    y_modelo,
    cv=cv_splits,
    method="predict",
)

decision_svm = cross_val_predict(
    modelo_svm,
    X_modelo,
    y_modelo,
    cv=cv_splits,
    method="decision_function",
)

metrics_svm = {
    "Accuracy": accuracy_score(y_modelo, pred_svm),
    "Balanced Accuracy": balanced_accuracy_score(y_modelo, pred_svm),
    "Precision relevant": precision_score(y_modelo, pred_svm),
    "Recall relevant": recall_score(y_modelo, pred_svm),
    "F1 relevant": f1_score(y_modelo, pred_svm),
}

display(pd.Series(metrics_svm, name="Linear SVM").round(3).to_frame())

print("Confusion matrix:")
print(confusion_matrix(y_modelo, pred_svm))


### Resultado esperado de Linear SVM

- Accuracy: **0.899**
- Balanced Accuracy: **0.890**
- Precision relevant: **0.938**
- Recall relevant: **0.914**
- F1 relevant: **0.926**
- Matriz de confusión: `[[45, 7], [10, 106]]`

SVM es ligeramente más conservador: produce **7 falsos positivos**, frente a 10 de Logistic Regression, y obtiene la mejor Balanced Accuracy.


## 10. Evaluación por plataforma

Las métricas globales no son suficientes porque X y YouTube tienen distribuciones de clase muy diferentes.

La evaluación por plataforma funciona como un diagnóstico adicional para detectar si el buen resultado se concentra en una sola fuente.


In [ ]:
def metricas_por_plataforma(predicciones, nombre):
    rows = []

    for plataforma in ["x", "youtube"]:
        mask = dataset_modelo_limpio["platform"].eq(plataforma).to_numpy()
        y_real = y_modelo[mask]
        y_pred = predicciones[mask]

        rows.append(
            {
                "Modelo": nombre,
                "Plataforma": plataforma,
                "N": int(mask.sum()),
                "Accuracy": accuracy_score(y_real, y_pred),
                "Balanced Accuracy": balanced_accuracy_score(y_real, y_pred),
                "Precision relevant": precision_score(
                    y_real, y_pred, zero_division=0
                ),
                "Recall relevant": recall_score(
                    y_real, y_pred, zero_division=0
                ),
                "F1 relevant": f1_score(
                    y_real, y_pred, zero_division=0
                ),
            }
        )

    return pd.DataFrame(rows)

por_plataforma = pd.concat(
    [
        metricas_por_plataforma(pred_logreg, "Logistic Regression"),
        metricas_por_plataforma(pred_svm, "Linear SVM"),
    ],
    ignore_index=True,
)

display(
    por_plataforma.round(3)
)


### Lectura de los resultados por plataforma

**Logistic Regression**
- X: Accuracy 0.918, Balanced Accuracy 0.567, F1 0.957
- YouTube: Accuracy 0.887, Balanced Accuracy 0.886, F1 0.846

**Linear SVM**
- X: Accuracy 0.907, Balanced Accuracy 0.639, F1 0.950
- YouTube: Accuracy 0.887, Balanced Accuracy 0.867, F1 0.833

El desempeño de X debe interpretarse con cautela: solo existen **6 negativos**.  
En Logistic Regression se reconoce correctamente 1 de esos 6; Linear SVM reconoce 2.

Por ello, el siguiente incremento de calidad más importante no es cambiar de algoritmo, sino **ampliar el conjunto de hard negatives de X**.


## 11. Comparación final

La comparación se realiza sobre el mismo dataset limpio y usando las mismas particiones de validación.


In [ ]:
comparacion_modelos = pd.DataFrame(
    {
        "Platform-only baseline": baseline_metrics,
        "Logistic Regression": metrics_logreg,
        "Linear SVM": metrics_svm,
    }
).T.round(3)

display(comparacion_modelos)


### Interpretación

**Logistic Regression**
- mejor Accuracy;
- mejor Recall;
- mejor F1;
- preferible si el costo principal es perder comentarios relevantes.

**Linear SVM**
- mejor Balanced Accuracy;
- mejor Precision;
- menos falsos positivos;
- preferible si el costo principal es dejar pasar comentarios irrelevantes a las etapas de sentimiento y tópicos.

Como el filtro está situado **antes** del resto de la arquitectura, para una primera integración se propone **Linear SVM como candidato v1**, sin considerar todavía cerrada la selección definitiva.


## 12. Análisis de errores

Se revisan los casos en los que ambos modelos fallan. Estos ejemplos son más informativos que la métrica agregada porque permiten descubrir los límites reales del criterio aprendido.

Los errores observados incluyen:

- menciones a Banamex que realmente hablan de promociones u otros servicios;
- referencias a apps de otros bancos;
- lenguaje digital ambiguo;
- comentarios relevantes muy cortos, agresivos o poco estructurados;
- expresiones que requieren contexto externo para interpretarse.

Esto confirma que una regla de palabras clave (`app`, `Banamex`, `tarjeta`, etc.) sería insuficiente.


In [ ]:
errores = dataset_modelo_limpio[
    [
        "record_id",
        "platform",
        "text",
        "etiqueta_texto",
        "target",
    ]
].copy()

errores["pred_logreg"] = pred_logreg
errores["pred_svm"] = pred_svm

errores["error_logreg"] = errores["target"] != errores["pred_logreg"]
errores["error_svm"] = errores["target"] != errores["pred_svm"]

print("Errores Logistic Regression:", int(errores["error_logreg"].sum()))
print("Errores Linear SVM:", int(errores["error_svm"].sum()))
print()

errores_ambos = errores[
    errores["error_logreg"] & errores["error_svm"]
].copy()

display(
    errores_ambos[
        [
            "platform",
            "text",
            "etiqueta_texto",
            "pred_logreg",
            "pred_svm",
        ]
    ]
)


## 13. Entrenamiento del modelo candidato y handoff

Una vez comparados los modelos mediante predicciones *out-of-fold*, el modelo seleccionado para integración se entrena con **todos los 168 registros claros y únicos**.

Se guarda el pipeline completo, no solamente el clasificador. Esto es importante porque el artefacto incluye el `StandardScaler` y garantiza que inferencia y entrenamiento utilicen la misma transformación.

El modelo espera como entrada **embeddings de 384 dimensiones generados con el mismo perfil `minilm-verbatim-v1`**.


In [ ]:
modelo_relevancia_final = clone(modelo_svm)
modelo_relevancia_final.fit(X_modelo, y_modelo)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "relevance_filter_minilm_linearsvm_v1.joblib"
METADATA_PATH = MODEL_DIR / "relevance_filter_minilm_linearsvm_v1.metadata.json"

joblib.dump(
    modelo_relevancia_final,
    MODEL_PATH,
)

metadata = {
    "model_name": "relevance_filter_minilm_linearsvm_v1",
    "task": "binary_relevance_classification",
    "label_mapping": {
        "0": "irrelevant",
        "1": "relevant",
    },
    "classifier": "LinearSVC",
    "pipeline": [
        "StandardScaler",
        "LinearSVC(class_weight='balanced')",
    ],
    "dataset_release_id": resolved["dataset_release_id"],
    "embedding_release_id": resolved["embedding_release_id"],
    "embedding_profile": PROFILE,
    "embedding_dimensions": int(X_modelo.shape[1]),
    "training_records": int(len(dataset_modelo_limpio)),
    "training_class_counts": {
        str(k): int(v)
        for k, v in dataset_modelo_limpio[
            "etiqueta_texto"
        ].value_counts().items()
    },
    "training_platform_counts": {
        str(k): int(v)
        for k, v in dataset_modelo_limpio[
            "platform"
        ].value_counts().items()
    },
    "cross_validation": {
        "folds": 5,
        "stratification": "platform + target",
        "random_state": 42,
        "metrics": {
            k: round(float(v), 6)
            for k, v in metrics_svm.items()
        },
    },
    "known_limitations": [
        "Only 6 clear irrelevant examples from X are available.",
        "Ground truth was created with a text-only criterion.",
        "Uncertain labels were excluded from binary training.",
        "Additional independent annotation is recommended before production use.",
    ],
    "python_version": sys.version.split()[0],
    "scikit_learn_version": sklearn.__version__,
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Modelo guardado en:")
print(MODEL_PATH)
print()
print("Metadata guardada en:")
print(METADATA_PATH)


### Interfaz mínima para integración

La capa productiva no necesita volver a conocer el proceso de entrenamiento. Solo requiere:

1. obtener el embedding MiniLM del comentario;
2. cargar el artefacto `.joblib`;
3. aplicar `predict`;
4. conservar únicamente `relevant` para las etapas posteriores.

`decision_function` puede conservarse como señal de confianza relativa. En una iteración posterior puede utilizarse para crear una zona de revisión manual alrededor del límite de decisión.


In [ ]:
def classify_relevance_from_embedding(embedding, model=modelo_relevancia_final):
    vector = np.asarray(embedding, dtype=float).reshape(1, -1)

    if vector.shape[1] != 384:
        raise ValueError(
            f"Se esperaban 384 dimensiones y se recibieron {vector.shape[1]}"
        )

    prediction = int(model.predict(vector)[0])
    decision_score = float(model.decision_function(vector)[0])

    return {
        "prediction": prediction,
        "label": "relevant" if prediction == 1 else "irrelevant",
        "decision_score": decision_score,
    }


## 14. Limitaciones y próximos pasos

El experimento demuestra viabilidad, pero no debe interpretarse todavía como una validación final de producción.

### Limitaciones actuales

1. **Pocos negativos de X.** Solo 6 ejemplos claros limitan la estabilidad de las métricas de esa plataforma.
2. **Ground truth limitado.** La auditoría fue construida con criterio *text-only* y conviene incorporar una segunda revisión independiente.
3. **Casos ambiguos.** Los `uncertain` no forman parte del entrenamiento binario; en producción podrían convertirse en una zona de revisión.
4. **Distribución cambiante.** Nuevos periodos pueden contener temas, productos o estilos de escritura diferentes.

### Próximos pasos recomendados

- aumentar específicamente los **hard negatives de X**;
- realizar una segunda anotación independiente sobre una muestra y medir acuerdo;
- versionar modelo, dataset, perfil de embeddings y métricas;
- registrar volumen de `relevant`, `irrelevant` y scores de decisión por periodo;
- monitorear cambios de distribución;
- integrar el filtro antes de sentimiento y modelado de temas;
- reentrenar o recalibrar cuando exista suficiente nueva evidencia.


## 15. Conclusión

Se construyó un filtro de relevancia para resolver una diferencia estructural entre las fuentes del proyecto: las reseñas de App Store y Google Play ya pertenecen a la app por definición, mientras que X y YouTube necesitan una validación adicional.

El flujo de trabajo fue:

**auditoría del ground truth → asociación con embeddings MiniLM → eliminación de ambigüedad y duplicados → baseline de plataforma → validación cruzada → Logistic Regression → Linear SVM → análisis de errores → selección de candidato**

Sobre 168 comentarios claros y únicos, ambos modelos semánticos superaron al baseline de plataforma. Logistic Regression obtuvo el mayor F1 y recall; Linear SVM obtuvo la mayor Balanced Accuracy, mayor precisión y menos falsos positivos.

Esto permite sostener que **el enfoque embeddings + clasificador lineal es técnicamente viable para la capa de relevancia**.

Para la primera integración se propone Linear SVM como candidato v1, manteniendo como condición de mejora la ampliación de ejemplos irrelevantes de X.

### Resultado arquitectónico

**App Store / Google Play → sin prefiltrado adicional**

**X / YouTube → MiniLM → filtro de relevancia → solo relevantes**

Después del filtro, las reseñas seleccionadas pueden alimentar las dos rutas aprobadas de la arquitectura:

- modelado de temas;
- análisis de sentimiento.

De esta forma el filtro no es un análisis aislado: funciona como una capa de calidad que protege los componentes posteriores y, finalmente, la información presentada en el dashboard.
